In [1]:
import pandas as pd

# Definir ruta del archivo
file_path = r'F:\Presentar\ORIGINAL.xlsx'  # Ruta actualizada

# Lista de columnas que serán consideradas como string
string_columns = [
    'NIVEL_GOB', 'SECTOR_GRAN', 'SECTOR', 'PLIEGO', 'EJECUTORA', 
    'SEC_EJEC', 'FUENTE', 'RUBRO', 'CATEGORIA_GTO', 'TIPO_TRANSACCION',
    'GENERICA', 'SUBGENERICA', 'SUBGENERICA_DET', 'ESPECIFICA', 
    'ESPECIFICA_DET', 'CLASIFICADOR_GASTO', 'PARTIDA_GASTO', 
    'SUBPARTIDA_GASTO', 'CONCEPTO', 'ETIQUETA_ART_9_13'
]

# Leer el archivo Excel, asegurando que las columnas especificadas sean string
dtype_spec = {col: str for col in string_columns}

# Cargar solo la hoja requerida
df = pd.read_excel(file_path, sheet_name='PRESENTAR', dtype=dtype_spec)

# Convertir el resto de columnas a enteros
for col in df.columns:
    if col not in string_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Guardar el DataFrame parcial en formato CSV separado por '|'
df.to_csv('dataframe_procesado.csv', sep='|', index=False, encoding='utf-8-sig')

# Mostrar las primeras filas para confirmar
print("Primeras filas del DataFrame procesado:")
print(df.head())

# Confirmar tipos de datos
print("\nTipos de datos por columna:")
print(df.dtypes)


Primeras filas del DataFrame procesado:
              NIVEL_GOB   SECTOR_GRAN                             SECTOR  \
0  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
1  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
2  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
3  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
4  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   

                                              PLIEGO  \
0  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
1  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
2  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
3  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
4  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   

                                           EJECUTORA SEC_EJEC  \
0  001. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...     1760   
1  001. ORGANISMO DE ESTUDIOS Y DISE

In [2]:
# Convertir las columnas de texto (object) a tipo string
for col in string_columns:
    df[col] = df[col].astype('string')

# Verificar los tipos nuevamente
print(df.dtypes)


NIVEL_GOB             string[python]
SECTOR_GRAN           string[python]
SECTOR                string[python]
PLIEGO                string[python]
EJECUTORA             string[python]
SEC_EJEC              string[python]
FUENTE                string[python]
RUBRO                 string[python]
CATEGORIA_GTO         string[python]
TIPO_TRANSACCION      string[python]
GENERICA              string[python]
SUBGENERICA           string[python]
SUBGENERICA_DET       string[python]
ESPECIFICA            string[python]
ESPECIFICA_DET        string[python]
CLASIFICADOR_GASTO    string[python]
PARTIDA_GASTO         string[python]
SUBPARTIDA_GASTO      string[python]
CONCEPTO              string[python]
ETIQUETA_ART_9_13     string[python]
PIA_2024                       int32
PIA_2023                       int32
PIA_2022                       int32
PIA_2021                       int32
PIA_2020                       int32
PIM_2024                       int32
PIM_2023                       int32
P

In [3]:
#ANULAMOS DINAMIZACION DE COLUMNAS 

In [4]:
# Lista de columnas a anular (dinamizar)
columns_to_unpivot = [
    'PIA_2024', 'PIA_2023', 'PIA_2022', 'PIA_2021', 'PIA_2020',
    'PIM_2024', 'PIM_2023', 'PIM_2022', 'PIM_2021', 'PIM_2020',
    'DEV_ANUAL_2024', 'DEV_ANUAL_2023', 'DEV_ANUAL_2022', 
    'DEV_ANUAL_2021', 'DEV_ANUAL_2020'
]

# Realizar el proceso "melt" para anular la dinamización
df_unpivot = df.melt(
    id_vars=[col for col in df.columns if col not in columns_to_unpivot],
    value_vars=columns_to_unpivot,
    var_name="Variable",  # Columna con los nombres originales de las columnas
    value_name="Valor"    # Columna con los valores correspondientes
)

# Crear la columna "Año" extrayendo el año de la columna "Variable"
df_unpivot['Año'] = df_unpivot['Variable'].str.extract(r'(\d{4})')

# Mostrar las primeras filas del nuevo DataFrame
print(df_unpivot.head())


              NIVEL_GOB   SECTOR_GRAN                             SECTOR  \
0  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
1  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
2  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
3  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
4  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   

                                              PLIEGO  \
0  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
1  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
2  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
3  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
4  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   

                                           EJECUTORA SEC_EJEC  \
0  001. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...     1760   
1  001. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...     1760   
2  001. ORG

In [5]:
import numpy as np

# Condición específica para "6.0 CPMP"
condicion_cpmp = (df_unpivot['Año'] == '2024') & (
    df_unpivot['PLIEGO'].isin(["026. M. DE DEFENSA", "007. M. DEL INTERIOR"])
) & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.2\.1\.1\.1\.99(\.\d+)*$') |
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.2\.2\.2\.1\.99(\.\d+)*$') |
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.5\.5\.1\.1\.99(\.\d+)*$')
)

# Condición específica para "7.0 BONOS DE RECONOCIMIENTO"
condicion_bonos_reconocimiento = (df_unpivot['Año'] == '2024') & (
    df_unpivot['PLIEGO'] == "095. OFICINA DE NORMALIZACION PREVISIONAL-ONP"
) & df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.8\.1(\.\d+)*$')

# Condición específica para "5.0 INDEMNIZACIONES/COMPENSACIONES"
condicion_indemnizaciones = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.5\.5\.2(\.\d+)*$')
)

# Condición específica para "4.0 NEGOCIACIONES COLECTIVAS"
condicion_negociaciones = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.5\.6(\.\d+)*$')
)

# Condición específica para "3.1 OTROS TIPOS DE CONTRATO DE PERSONAL"
condicion_otros_contratos = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.7\.5(\.\d+)*$') |
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.7\.12(\.\d+)*$')
) & ~df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.7\.5\.9(\.\d+)*$')

# Condición específica para "3.0 LOCACIÓN DE SERVICIOS"
condicion_locacion_servicios = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.9\.1\.1(\.\d+)*$')
)

# Condición específica para "2.1 JUDICIALES PENSIONISTAS"
condicion_judiciales_pensionistas = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.2\.1\.2(\.\d+)*$')
)

# Condición específica para "2.0 PENSIONISTAS"
condicion_pensionistas = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.2(\.\d+)*$')
)

# Condición específica para "1.2 JUDICIALES ACTIVOS"
condicion_judiciales = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.1\.5(\.\d+)*$')
)

# Condición específica para "1.1 CAS"
condicion_cas = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.1\.1\.13(\.\d+)*$') |
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.1\.1\.9\.1\.4(\.\d+)*$') |
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.1\.3\.1\.1\.15(\.\d+)*$')
)

# Condición general para "1.0 ACTIVOS"
condicion_activos = (df_unpivot['Año'] == '2024') & (
    df_unpivot['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.1(\.\d+)*$')
)

# Aplicar las condiciones en orden de prioridad
df_unpivot['Partida_Historica'] = np.select(
    [
        condicion_cpmp,                  # Nueva condición para "6.0 CPMP"
        condicion_bonos_reconocimiento, # Condición para "7.0 BONOS DE RECONOCIMIENTO"
        condicion_indemnizaciones,       # Condición para "5.0 INDEMNIZACIONES/COMPENSACIONES"
        condicion_negociaciones,         # Condición para "4.0 NEGOCIACIONES COLECTIVAS"
        condicion_otros_contratos,       # Condición para "3.1 OTROS TIPOS DE CONTRATO DE PERSONAL"
        condicion_locacion_servicios,    # Condición para "3.0 LOCACIÓN DE SERVICIOS"
        condicion_judiciales_pensionistas, # Condición para "2.1 JUDICIALES PENSIONISTAS"
        condicion_pensionistas,          # Condición para "2.0 PENSIONISTAS"
        condicion_judiciales,            # Condición para "1.2 JUDICIALES ACTIVOS"
        condicion_cas,                   # Condición para "1.1 CAS"
        condicion_activos                # Condición general para "1.0 ACTIVOS"
    ],
    [
        "6.0 CPMP",                      # Valor para CPMP
        "7.0 BONOS DE RECONOCIMIENTO",   # Valor para bonos de reconocimiento
        "5.0 INDEMNIZACIONES/COMPENSACIONES",  # Valor para indemnizaciones
        "4.0 NEGOCIACIONES COLECTIVAS",       # Valor para negociaciones colectivas
        "3.1 OTROS TIPOS DE CONTRATO DE PERSONAL",  # Valor para otros contratos
        "3.0 LOCACIÓN DE SERVICIOS",          # Valor para locación de servicios
        "2.1 JUDICIALES PENSIONISTAS",        # Valor para judiciales pensionistas
        "2.0 PENSIONISTAS",                   # Valor para pensionistas
        "1.2 JUDICIALES ACTIVOS",             # Valor para judiciales activos
        "1.1 CAS",                            # Valor para CAS
        "1.0 ACTIVOS"                         # Valor general para activos
    ],
    default=None                             # Valor por defecto si no se cumplen las condiciones
)

# Mostrar las filas actualizadas con Partida_Historica
print(df_unpivot[df_unpivot['Partida_Historica'].notnull()].head(10))



              NIVEL_GOB   SECTOR_GRAN                             SECTOR  \
0  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
1  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
2  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
3  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
4  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
5  1. GOBIERNO NACIONAL  SEDE CENTRAL                        26. DEFENSA   
6  1. GOBIERNO NACIONAL  SEDE CENTRAL                      10. EDUCACION   
7  1. GOBIERNO NACIONAL  SEDE CENTRAL                      10. EDUCACION   
8  1. GOBIERNO NACIONAL  SEDE CENTRAL                      10. EDUCACION   
9  1. GOBIERNO NACIONAL  SEDE CENTRAL                      10. EDUCACION   

                                              PLIEGO  \
0  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
1  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT..

In [6]:
import numpy as np

# Aplicar condiciones para los años 2020 a 2023
def asignar_partida_historica_2020_2023(df):
    # Condición específica para "6.0 CPMP"
    condicion_cpmp = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['PLIEGO'].isin(["026. M. DE DEFENSA", "007. M. DEL INTERIOR"])
    ) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.2\.1\.1\.1\.99(\.\d+)*$') |
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.2\.2\.2\.1\.99(\.\d+)*$') |
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.5\.5\.1\.1\.99(\.\d+)*$')
    )
    
    # Condición específica para "7.0 BONOS DE RECONOCIMIENTO"
    condicion_bonos_reconocimiento = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['PLIEGO'] == "095. OFICINA DE NORMALIZACION PREVISIONAL-ONP"
    ) & df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.8\.1(\.\d+)*$')
    
    # Condición específica para "5.0 INDEMNIZACIONES/COMPENSACIONES"
    condicion_indemnizaciones = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.5\.5\.2(\.\d+)*$')
    )
    
    # Condición específica para "4.0 NEGOCIACIONES COLECTIVAS"
    condicion_negociaciones = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.5\.6(\.\d+)*$')
    )
    
    # Condición específica para "3.1 OTROS TIPOS DE CONTRATO DE PERSONAL"
    condicion_otros_contratos = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.7\.5(\.\d+)*$') |
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.7\.12(\.\d+)*$')
    ) & ~df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.7\.5\.9(\.\d+)*$')
    
    # Condición específica para "3.0 LOCACIÓN DE SERVICIOS"
    condicion_locacion_servicios = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.9\.1\.1(\.\d+)*$')
    )
    
    # Condición específica para "2.1 JUDICIALES PENSIONISTAS"
    condicion_judiciales_pensionistas = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.5\.5\.1\.2(\.\d+)*$')
    )
    
    # Condición específica para "2.0 PENSIONISTAS"
    condicion_pensionistas = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.2(\.\d+)*$')
    )
    
    # Condición específica para "1.2 JUDICIALES ACTIVOS"
    condicion_judiciales = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.5\.5\.1\.1(\.\d+)*$')
    )
    
    # Condición específica para "1.1 CAS"
    condicion_cas = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.8\.1\.1(\.\d+)*$') |
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.8\.1\.4(\.\d+)*$') |
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.3\.2\.8\.1\.2(\.\d+)*$')
    )
    
    # Condición general para "1.0 ACTIVOS"
    condicion_activos = (df['Año'].isin(['2020', '2021', '2022', '2023'])) & (
        df['CLASIFICADOR_GASTO'].str.match(r'^5\.2\.1(\.\d+)*$')
    )
    
    # Actualizar la columna existente "Partida_Historica"
    df.loc[condicion_cpmp, 'Partida_Historica'] = "6.0 CPMP"
    df.loc[condicion_bonos_reconocimiento, 'Partida_Historica'] = "7.0 BONOS DE RECONOCIMIENTO"
    df.loc[condicion_indemnizaciones, 'Partida_Historica'] = "5.0 INDEMNIZACIONES/COMPENSACIONES"
    df.loc[condicion_negociaciones, 'Partida_Historica'] = "4.0 NEGOCIACIONES COLECTIVAS"
    df.loc[condicion_otros_contratos, 'Partida_Historica'] = "3.1 OTROS TIPOS DE CONTRATO DE PERSONAL"
    df.loc[condicion_locacion_servicios, 'Partida_Historica'] = "3.0 LOCACIÓN DE SERVICIOS"
    df.loc[condicion_judiciales_pensionistas, 'Partida_Historica'] = "2.1 JUDICIALES PENSIONISTAS"
    df.loc[condicion_pensionistas, 'Partida_Historica'] = "2.0 PENSIONISTAS"
    df.loc[condicion_judiciales, 'Partida_Historica'] = "1.2 JUDICIALES ACTIVOS"
    df.loc[condicion_cas, 'Partida_Historica'] = "1.1 CAS"
    df.loc[condicion_activos, 'Partida_Historica'] = "1.0 ACTIVOS"
    
    return df

# Aplicar las reglas para los años 2020 a 2023
df_unpivot = asignar_partida_historica_2020_2023(df_unpivot)

# Mostrar resultados
print(df_unpivot.head())



              NIVEL_GOB   SECTOR_GRAN                             SECTOR  \
0  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
1  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
2  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
3  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
4  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   

                                              PLIEGO  \
0  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
1  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
2  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
3  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   
4  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...   

                                           EJECUTORA SEC_EJEC  \
0  001. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...     1760   
1  001. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYECT...     1760   
2  001. ORG

In [7]:
####################### EXPORTAMOS A CSV #########################

# Definir ruta de exportación
output_path = r'F:\Presentar\Final\partida_historica_final.csv'

# Exportar DataFrame a CSV con separador "|"
df_unpivot.to_csv(output_path, sep='|', index=False, encoding='utf-8-sig')

print(f"Archivo exportado exitosamente a: {output_path}")


Archivo exportado exitosamente a: F:\Presentar\Final\partida_historica_final.csv


In [8]:
# Consultar valores únicos en la columna "Variable"
valores_unicos = df_unpivot['Variable'].unique()
print("Valores únicos en la columna 'Variable':")
print(valores_unicos)
print(df_unpivot.columns)


Valores únicos en la columna 'Variable':
['PIA_2024' 'PIA_2023' 'PIA_2022' 'PIA_2021' 'PIA_2020' 'PIM_2024'
 'PIM_2023' 'PIM_2022' 'PIM_2021' 'PIM_2020' 'DEV_ANUAL_2024'
 'DEV_ANUAL_2023' 'DEV_ANUAL_2022' 'DEV_ANUAL_2021' 'DEV_ANUAL_2020']
Index(['NIVEL_GOB', 'SECTOR_GRAN', 'SECTOR', 'PLIEGO', 'EJECUTORA', 'SEC_EJEC',
       'FUENTE', 'RUBRO', 'CATEGORIA_GTO', 'TIPO_TRANSACCION', 'GENERICA',
       'SUBGENERICA', 'SUBGENERICA_DET', 'ESPECIFICA', 'ESPECIFICA_DET',
       'CLASIFICADOR_GASTO', 'PARTIDA_GASTO', 'SUBPARTIDA_GASTO', 'CONCEPTO',
       'ETIQUETA_ART_9_13', 'Variable', 'Valor', 'Año', 'Partida_Historica'],
      dtype='object')


In [9]:
#AÑADIMOS LO EJECUTADO POR AÑO

# Crear lista de años
años = ['2020', '2021', '2022', '2023', '2024']

# Crear ejecución anual (Ejec_202*) como nuevas filas
nuevas_filas = []

for año in años:
    # Filtrar PIM y DEV_ANUAL para el año correspondiente
    pim = df_unpivot[(df_unpivot['Año'] == año) & (df_unpivot['Variable'] == f'PIM_{año}')].copy()
    dev = df_unpivot[(df_unpivot['Año'] == año) & (df_unpivot['Variable'] == f'DEV_ANUAL_{año}')].copy()

    # Hacer un merge para combinar PIM y DEV_ANUAL
    ejec = pim.merge(dev, on=['Año', 'PLIEGO', 'CLASIFICADOR_GASTO'], suffixes=('_PIM', '_DEV'))

    # Calcular la ejecución anual evitando división por cero
    ejec['Valor'] = ejec.apply(
        lambda row: row['Valor_DEV'] / row['Valor_PIM'] if row['Valor_PIM'] != 0 else 0, axis=1
    )

    # Crear una nueva variable 'Ejec_202*'
    ejec['Variable'] = f'Ejec_{año}'
    
    # Seleccionar columnas relevantes
    nuevas_filas.append(ejec[['Año', 'PLIEGO', 'CLASIFICADOR_GASTO', 'Variable', 'Valor']])

# Concatenar las nuevas filas al DataFrame original
df_unpivot = pd.concat([df_unpivot] + nuevas_filas, ignore_index=True)

print("Ejecución anual calculada y agregada como nuevas variables.")
print(df_unpivot.head(10))




Ejecución anual calculada y agregada como nuevas variables.
              NIVEL_GOB   SECTOR_GRAN                             SECTOR  \
0  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
1  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
2  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
3  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
4  1. GOBIERNO NACIONAL  SEDE CENTRAL  01. PRESIDENCIA CONSEJO MINISTROS   
5  1. GOBIERNO NACIONAL  SEDE CENTRAL                        26. DEFENSA   
6  1. GOBIERNO NACIONAL  SEDE CENTRAL                      10. EDUCACION   
7  1. GOBIERNO NACIONAL  SEDE CENTRAL                      10. EDUCACION   
8  1. GOBIERNO NACIONAL  SEDE CENTRAL                      10. EDUCACION   
9  1. GOBIERNO NACIONAL  SEDE CENTRAL                      10. EDUCACION   

                                              PLIEGO  \
0  031. ORGANISMO DE ESTUDIOS Y DISEÑO DE PROYE

In [10]:
# Filtrar los valores que coincidan con el patrón "Ejec_202*"
valores_ejec_unicos = df_unpivot['Variable'].loc[df_unpivot['Variable'].str.contains(r'^Ejec_202\d{1}$', na=False)].unique()

# Mostrar los valores únicos encontrados
print("Valores únicos que coinciden con el patrón 'Ejec_202*':")
print(valores_ejec_unicos)

print(df_unpivot.columns)


Valores únicos que coinciden con el patrón 'Ejec_202*':
['Ejec_2020' 'Ejec_2021' 'Ejec_2022' 'Ejec_2023' 'Ejec_2024']
Index(['NIVEL_GOB', 'SECTOR_GRAN', 'SECTOR', 'PLIEGO', 'EJECUTORA', 'SEC_EJEC',
       'FUENTE', 'RUBRO', 'CATEGORIA_GTO', 'TIPO_TRANSACCION', 'GENERICA',
       'SUBGENERICA', 'SUBGENERICA_DET', 'ESPECIFICA', 'ESPECIFICA_DET',
       'CLASIFICADOR_GASTO', 'PARTIDA_GASTO', 'SUBPARTIDA_GASTO', 'CONCEPTO',
       'ETIQUETA_ART_9_13', 'Variable', 'Valor', 'Año', 'Partida_Historica'],
      dtype='object')


In [11]:
# Asegurar que 'Partida_Historica' sea string y 'Año' sea entero
df_unpivot['Partida_Historica'] = df_unpivot['Partida_Historica'].astype(str)
df_unpivot['Año'] = pd.to_numeric(df_unpivot['Año'], errors='coerce').fillna(0).astype(int)

# Definir la ruta de exportación
output_path = r'F:\Presentar\Final\Proy_2025.csv'

# Exportar el DataFrame a un archivo CSV
df_unpivot.to_csv(output_path, sep='|', index=False, encoding='utf-8-sig')

print(f"El archivo se ha guardado exitosamente en: {output_path}")


El archivo se ha guardado exitosamente en: F:\Presentar\Final\Proy_2025.csv


In [12]:
print(df_unpivot.dtypes)

NIVEL_GOB             string[python]
SECTOR_GRAN           string[python]
SECTOR                string[python]
PLIEGO                string[python]
EJECUTORA             string[python]
SEC_EJEC              string[python]
FUENTE                string[python]
RUBRO                 string[python]
CATEGORIA_GTO         string[python]
TIPO_TRANSACCION      string[python]
GENERICA              string[python]
SUBGENERICA           string[python]
SUBGENERICA_DET       string[python]
ESPECIFICA            string[python]
ESPECIFICA_DET        string[python]
CLASIFICADOR_GASTO    string[python]
PARTIDA_GASTO         string[python]
SUBPARTIDA_GASTO      string[python]
CONCEPTO              string[python]
ETIQUETA_ART_9_13     string[python]
Variable                      object
Valor                        float64
Año                            int32
Partida_Historica             object
dtype: object


In [30]:
import pandas as pd

# Definir la ruta del archivo
input_path = r'F:\Presentar\Final\Proy_2025.csv'

# Cargar el archivo CSV
df = pd.read_csv(input_path, sep='|', encoding='utf-8-sig')




C:\Users\Alons\AppData\Local\Temp\ipykernel_10692\1770586682.py:7: DtypeWarning: Columns (0,1,2,4,6,7,8,9,10,11,12,13,14,16,17,18,19,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path, sep='|', encoding='utf-8-sig')


Archivo cargado exitosamente. Primeras filas del DataFrame:


AttributeError: 'DataFrame' object has no attribute 'dtype'

In [32]:
print("Archivo cargado exitosamente. Primeras filas del DataFrame:")
print(df.dtypes)

Archivo cargado exitosamente. Primeras filas del DataFrame:
NIVEL_GOB              object
SECTOR_GRAN            object
SECTOR                 object
PLIEGO                 object
EJECUTORA              object
SEC_EJEC              float64
FUENTE                 object
RUBRO                  object
CATEGORIA_GTO          object
TIPO_TRANSACCION       object
GENERICA               object
SUBGENERICA            object
SUBGENERICA_DET        object
ESPECIFICA             object
ESPECIFICA_DET         object
CLASIFICADOR_GASTO     object
PARTIDA_GASTO          object
SUBPARTIDA_GASTO       object
CONCEPTO               object
ETIQUETA_ART_9_13      object
Variable               object
Valor                 float64
Año                     int64
Partida_Historica      object
dtype: object


In [ ]:
# Convertir todas las columnas a string, excepto 'Valor' y 'Año'
for col in df.columns:
    if col not in ['Valor', 'Año']:
        df[col] = df[col].astype(str)

# Asegurar que 'Valor' y 'Año' sean enteros
df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce').fillna(0).astype(int)
df['Año'] = pd.to_numeric(df['Año'], errors='coerce').fillna(0).astype(int)

# Dinamizar (despivotar) las columnas Variable y Valor
df_dinamizado = df.melt(
    id_vars=[col for col in df.columns if col not in ['Variable', 'Valor']],
    value_vars=['Valor'],
    var_name='Variable_Dinamizada',
    value_name='Valor_Dinamizado'
)

# Mostrar el DataFrame dinamizado
print("DataFrame dinamizado:")
print(df_dinamizado.dtypes)
print(df_dinamizado.head(2))

# Opcional: Verificar las columnas resultantes
print("Columnas resultantes en el DataFrame dinamizado:")
print(df_dinamizado.columns)


In [ ]:
# Definir la ruta de exportación
output_path = r'F:\Presentar\Final\Proy_2025.csv'

# Exportar el DataFrame a un archivo CSV
df_unpivot.to_csv(output_path, sep='|', index=False, encoding='utf-8-sig')

print(f"El archivo se ha guardado exitosamente en: {output_path}")
